[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aimldstejas/aibits-genai-notebooks/blob/main/ml/05-full-projects/ml-fullproj-leadpred.ipynb)

# Full Project: Credit Card Lead Prediction (Bank Cross-Sell)

*AIBits Academy · Machine Learning End To End · Full Project*

A complete binary classification pipeline at bank scale — 245,725 customers, one-hot encoding at scale, and a four-model bake-off where the "obvious" ensemble choice doesn't win.

**How to use this notebook:** run the cells top to bottom (Runtime → Run all). Each code cell is the same code you saw on the course page, so you can compare your output with the lesson. The graded exercises are at the end; try them before opening the solutions.

*Interactive animations and quiz cards stay on the course page.*

*Small numeric differences from the lesson page are normal: library versions, random seeds and dataset copies change the last digits. The conclusions should agree.*

## Setup

In [ ]:
# Fetch the lesson's dataset(s) into the working folder
import os, io, zipfile, urllib.request, urllib.parse

DATA_BASE = "https://raw.githubusercontent.com/aimldstejas/aibits-genai-notebooks/main/ml/data/"   # course copies live in the notebooks repo
for f in ['train_data_credit_card.csv']:
    if not os.path.exists(f):
        urllib.request.urlretrieve(DATA_BASE + urllib.parse.quote(f), f)
        print('downloaded', f)

In [ ]:
# Imports used throughout this project
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, classification_report, mean_squared_error, mean_absolute_error, r2_score)

> **Business Problem**
>
> A retail bank wants to cross-sell credit cards to its existing customer base. Calling all 245,725 customers is expensive and annoys the majority who aren't interested; the bank wants a model that ranks customers by likelihood of becoming a "lead" (`Is_Lead` = 1), so outreach can focus on the customers most worth calling.

> **Dataset**
>
> **245,725 customers, 9 features after dropping ID, binary target.** Features: Gender, Age, Region_Code (35 regions), Occupation (Other/Salaried/Self_Employed/Entrepreneur), Channel_Code (X1–X4), Vintage (months as customer), Credit_Product (Yes/No, 29,325 missing), Avg_Account_Balance, Is_Active. Target `Is_Lead`: 23.7% positive base rate.

## Step 1 — Load, Clean, De-duplicate

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("train_data_credit_card.csv")
df.drop("ID", axis=1, inplace=True)
print(df.shape)
print(df.isnull().sum())
print("Duplicate rows:", df.duplicated().sum())

In [ ]:
df.drop_duplicates(inplace=True)

# Credit_Product is categorical (Yes/No) -- impute with the mode, not the mean
mode_val = df["Credit_Product"].mode().values[0]
df["Credit_Product"] = df["Credit_Product"].fillna(mode_val)
print(df.isnull().sum().sum(), "missing values remaining")

## Step 2 — Key EDA Findings

Before modelling, a quick pass across demographics and target correlates surfaces the strongest signals:

- The target is imbalanced but not severely so: **23.7% positive** — no oversampling is strictly required, though it's worth testing.
- Entrepreneurs carry the highest average account balance of any occupation group, well above Salaried and Self-Employed customers.
- Salaried customers cluster heavily in the 24–36 age range; Self-Employed and Entrepreneur segments skew markedly older (37–61).
- Higher `Avg_Account_Balance` correlates with higher `Vintage` — longer-tenured customers have had more time to accumulate balance, an unsurprising but useful confirmation that the features behave sensibly.

## Step 3 — Encode and Standardise

In [ ]:
from scipy.stats import zscore

# One-hot encode all categorical columns
df = pd.get_dummies(df, columns=["Region_Code", "Gender", "Occupation",
                                   "Channel_Code", "Credit_Product", "Is_Active"])
print(df.shape)   # 52 columns after one-hot encoding a 35-level Region_Code

X = df.drop(columns=["Is_Lead"])
y = df["Is_Lead"]
X = zscore(X.astype(float))   # standardise every feature to mean 0, std 1

## Step 4 — Train/Test Split and a Four-Model Bake-Off

In [ ]:
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.metrics import accuracy_score

X_train, X_test, y_train, y_test = train_test_split(X, y, train_size=0.70, random_state=100)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

models = {
    "Logistic Regression": LogisticRegression(),
    "Decision Tree":       DecisionTreeClassifier(criterion="entropy", max_depth=6, random_state=0),
    "Random Forest":       RandomForestClassifier(),
    "AdaBoost":            AdaBoostClassifier(),
}
for name, model in models.items():
    model.fit(X_train, y_train)
    acc = accuracy_score(y_test, model.predict(X_test))
    print(f"{name:22s} accuracy: {acc:.4f}")

## Step 5 — Hyperparameter Tuning: Not Every Model Improves

**Heads-up:** these grid searches fit hundreds of models on 170,000 rows. On Colab's free CPU expect this cell to run for a good while (10-20 minutes). You can shrink the grids to try it quickly.

In [ ]:
# Decision Tree: GridSearchCV improves it
dt_params = {"max_depth": [10, 20], "max_leaf_nodes": [250, 270], "min_samples_split": [5, 8]}
dt_grid = GridSearchCV(DecisionTreeClassifier(criterion="entropy", random_state=0), dt_params, cv=10)
dt_grid.fit(X_train, y_train)
print("Decision Tree best CV score:", dt_grid.best_score_, dt_grid.best_params_)
print("Decision Tree tuned test score:", dt_grid.score(X_test, y_test))

# Random Forest: GridSearchCV makes it WORSE
rf_params = {"max_depth": [2, 5, 10, 30], "min_samples_split": [5, 10, 30],
             "min_impurity_decrease": [1.0, 2.0], "n_estimators": [20, 40], "random_state": [34, 40]}
rf_grid = GridSearchCV(RandomForestClassifier(), rf_params, cv=5)
rf_grid.fit(X_train, y_train)
print("Random Forest best CV score:", rf_grid.best_score_, rf_grid.best_params_)

> **A Wider Grid Isn't Always a Better Grid**
>
> The Decision Tree's grid search found genuinely better settings (79.12% vs 78.60% untuned). The Random Forest's grid search found *worse* settings (76.24% vs 77.52% untuned) — because the grid included `min_impurity_decrease` values (1.0, 2.0) that are enormous on Gini's 0–1 scale, effectively forbidding almost every split and forcing every tree in the forest to stay extremely shallow. GridSearchCV will happily return the "best" combination *within the grid you gave it* — it has no way to know the grid itself was badly specified. The fix here is domain knowledge about what a sane `min_impurity_decrease` range looks like (typically 0.0–0.01), not a bigger search.

## Final Model Comparison

| Model | Default Accuracy | Tuned Accuracy | Verdict |
|---|---|---|---|
| Logistic Regression | 77.71% | 77.62% (CV) | Tuning made no real difference |
| **Decision Tree** | 78.60% | **79.12%** | **Best overall — and interpretable** |
| Random Forest | 77.52% | 76.24% (CV, worse) | Badly-specified grid hurt it |
| AdaBoost | 78.22% | 78.25% | Marginal improvement |

A single, well-tuned Decision Tree (max_depth=20, max_leaf_nodes=250, min_samples_split=5) beat every ensemble tested — a genuinely surprising result that only shows up when every model is tuned fairly, rather than comparing default Random Forest against a hand-picked Decision Tree depth.

## Visualizing the Bake-Off

Default vs. tuned accuracy for all four models — Decision Tree is the only one that clearly improves with tuning; Random Forest's badly-specified grid actively makes it worse.

## Key Business Takeaways

- At 245K+ rows and 53 columns post-encoding, the pipeline is dominated by clean preprocessing (mode-imputation of a categorical column, not mean; one-hot encoding at scale) more than by exotic modelling.
- A well-tuned simple model can beat a poorly-tuned complex one — the ensemble "upgrade" from Decision Tree to Random Forest is not automatic and must be earned through equally careful tuning.
- The interpretability bonus is free here: a tuned Decision Tree gives the bank's marketing team explainable "if Vintage > X and Credit_Product = No and Occupation = Salaried then..." rules to justify why a customer was called, alongside the best accuracy of the four models tested.

## Practice Questions

---
## Graded exercises

Each exercise has a **starter cell** you complete and a **check cell** that prints ✅ or ❌. The solution is folded away underneath — try first.

In [ ]:
# --- self-check helper (used by the exercises) ---------------------------------------------
def check(name, ok):
    print(("\u2705 " if ok else "\u274c ") + name)


### Exercise 1 · Easy · The base rate

Store in `pos_rate` the fraction of customers who are leads (`y`). It is the accuracy floor: always predicting "not a lead" scores `1 - pos_rate`.

In [ ]:
pos_rate = None   # TODO


In [ ]:
try:
    check("about 23.7%", abs(pos_rate - 0.237) < 0.005)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
pos_rate = float(y.mean())

```

</details>

### Exercise 2 · Medium · Does the tree beat the naive baseline?

Using the fitted `models["Decision Tree"]`, store its test accuracy in `dt_acc`, the naive baseline accuracy in `naive_acc` (majority class in `y_test`), and `beats_naive` = `dt_acc > naive_acc`.

In [ ]:
dt_acc = naive_acc = beats_naive = None   # TODO


In [ ]:
try:
    check("naive is about 76%", 0.74 < naive_acc < 0.78)
    check("tree is better", beats_naive is True)
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
dt_acc = accuracy_score(y_test, models["Decision Tree"].predict(X_test))
naive_acc = max(y_test.mean(), 1 - y_test.mean())
beats_naive = bool(dt_acc > naive_acc)

```

</details>

### Exercise 3 · Stretch · What does the tree rely on?

Store the names of the three most important features of the fitted decision tree in `top3` (most important first). Column names come from the encoded frame: `df.drop(columns=["Is_Lead"]).columns`.

In [ ]:
top3 = None   # TODO


In [ ]:
try:
    cols = list(df.drop(columns=["Is_Lead"]).columns)
    check("three names", len(top3) == 3 and all(c in cols for c in top3))
    imp = models["Decision Tree"].feature_importances_
    check("ordered by importance", imp[cols.index(top3[0])] >= imp[cols.index(top3[1])] >= imp[cols.index(top3[2])])
except Exception as e:
    print("\u274c Not ready yet (" + type(e).__name__ + ") - complete the starter cell above, then run this again.")


<details><summary><b>Show solution</b></summary>

```python
cols = list(df.drop(columns=["Is_Lead"]).columns)
imp = models["Decision Tree"].feature_importances_
top3 = [cols[i] for i in imp.argsort()[::-1][:3]]

```

</details>

---
*Back to the course: **Machine Learning End To End → Full Project: Credit Card Lead Prediction (Bank Cross-Sell)**.*